In [1]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

In [2]:
import os
import kagglehub
import glob
from sklearn.model_selection import train_test_split
path = kagglehub.dataset_download("greatgamedota/culane")





Using Colab cache for faster access to the 'culane' dataset.


In [3]:
print(path)

/kaggle/input/culane


In [4]:


images = sorted(glob.glob(os.path.join(path, '**','*.jpg'),recursive=True))
masks = sorted(glob.glob(os.path.join(path, '**','*.png'),recursive=True))

print(len(images))


18233


In [5]:
train_images, test_images, train_masks, test_masks = train_test_split(images, masks, test_size=0.2, random_state=42)
val_images, test_images, val_masks, test_masks = train_test_split(test_images, test_masks, test_size=0.5, random_state=42)

print(f'Train: {len(train_images)}, Val: {len(val_images)}, Test: {len(test_images)}')

Train: 14586, Val: 1823, Test: 1824


In [6]:
def preprocess_data(img_path, msk_path, target_size=(256,256)):
  img = cv2.imread(img_path)
  img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
  img = cv2.resize(img, target_size)
  img = img / 255.0

  mask = cv2.imread(msk_path, cv2.IMREAD_GRAYSCALE)
  mask = cv2.resize(mask, target_size, interpolation=cv2.INTER_NEAREST)

  mask = mask.reshape((target_size[0], target_size[1], 1))

  return img, mask

In [7]:
def data_generator(img_path, msk_path, batch_size=16, target_size=(256,256)):
  while True:
    for i in range(0, len(img_path), batch_size):
      batch_img_path = img_path [i : i + batch_size]
      batch_msk_path = msk_path [i : i + batch_size]

      batch_x = []
      batch_y = []

      for img_p, msk_p in zip(batch_img_path, batch_msk_path):
        x, y = preprocess_data(img_p, msk_p, target_size)
        batch_x.append(x)
        batch_y.append(y)

      yield np.array(batch_x), np.array(batch_y)

In [8]:
train_gen = data_generator(train_images, train_masks, batch_size=16)
val_gen = data_generator(val_images, val_masks, batch_size=16)

x, y = next(train_gen)
print(f'Batch images shape: {x.shape}')
print(f'Batch images shape: {y.shape}')

Batch images shape: (16, 256, 256, 3)
Batch images shape: (16, 256, 256, 1)


In [9]:
from tensorflow.keras import layers, models
def build_model(input_shape=(256,256,3)):
  inputs = layers.Input(input_shape)

  #Encode
  c1 = layers.Conv2D(64,(3,3), activation='relu', padding='same')(inputs)

  c1 = layers.Conv2D(64, (3,3), activation='relu', padding='same')(c1)
  p1 = layers.MaxPooling2D((2,2))(c1)

  c2 = layers.Conv2D(128,(3,3), activation='relu', padding='same')(p1)
  c2 = layers.Conv2D(128,(3,3), activation='relu', padding='same')(c2)
  p2 = layers.MaxPooling2D((2,2))(c2)

  c3 = layers.Conv2D(256,(3,3), activation='relu', padding='same')(p2)
  c3 = layers.Conv2D(256,(3,3), activation='relu', padding='same')(c3)

  #decode
  u4 = layers.UpSampling2D((2,2))(c3)
  u4 = layers.concatenate([u4, c2])

  c4 = layers.Conv2D(128,(3,3), activation='relu', padding='same')(u4)

  u5 = layers.UpSampling2D((2,2))(c4)
  u5 = layers.concatenate([u5,c1])
  c5 = layers.Conv2D(64,(3,3), activation='relu', padding='same')(u5)

  output = layers.Conv2D(1,(1,1), activation='sigmoid')(c5)

  model = models.Model(inputs=[inputs], outputs=[output])
  return model

model = build_model()
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 256, 256,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 256, 256,  │      1,792 │ input_layer[0][0] │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 256, 256,  │     36,928 │ conv2d[0][0]      │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 128, 128,  │          0 │ conv2d_1[0][0]    │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 128, 128,  │     73,856 │ max_pooling2d[0]… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 128, 128,  │    147,584 │ conv2d_2[0][0]    │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 64, 64,    │          0 │ conv2d_3[0][0]    │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 64, 64,    │    295,168 │ max_pooling2d_1[… │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_5 (Conv2D)   │ (None, 64, 64,    │    590,080 │ conv2d_4[0][0]    │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ up_sampling2d       │ (None, 128, 128,  │          0 │ conv2d_5[0][0]    │
│ (UpSampling2D)      │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 128, 128,  │          0 │ up_sampling2d[0]… │
│ (Concatenate)       │ 384)              │            │ conv2d_3[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_6 (Conv2D)   │ (None, 128, 128,  │    442,496 │ concatenate[0][0] │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ up_sampling2d_1     │ (None, 256, 256,  │          0 │ conv2d_6[0][0]    │
│ (UpSampling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 256, 256,  │          0 │ up_sampling2d_1[… │
│ (Concatenate)       │ 192)              │            │ conv2d_1[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_7 (Conv2D)   │ (None, 256, 256,  │    110,656 │ concatenate_1[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_8 (Conv2D)   │ (None, 256, 256,  │         65 │ conv2d_7[0][0]    │
│                     │ 1)                │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 1,698,625 (6.48 MB)

 Trainable params: 1,698,625 (6.48 MB)

 Non-trainable params: 0 (0.00 B)

In [10]:
steps_per_epoch = len(train_images) // 16
val_steps = len(val_images) // 16

history = model.fit(
    train_gen,
    validation_data=val_gen,
    steps_per_epoch=steps_per_epoch,
    validation_steps=val_steps,
    epochs=20
    )

Epoch 1/20
911/911 ━━━━━━━━━━━━━━━━━━━━ 544s 548ms/step - accuracy: 0.9722 - loss: 0.1892 - val_accuracy: 0.9728 - val_loss: 0.1723
Epoch 2/20
911/911 ━━━━━━━━━━━━━━━━━━━━ 388s 426ms/step - accuracy: 0.9613 - loss: 0.2648 - val_accuracy: 0.9751 - val_loss: 0.4919
Epoch 3/20
911/911 ━━━━━━━━━━━━━━━━━━━━ 382s 419ms/step - accuracy: 0.9751 - loss: 0.3679 - val_accuracy: 0.9752 - val_loss: 0.3016
Epoch 4/20
911/911 ━━━━━━━━━━━━━━━━━━━━ 391s 429ms/step - accuracy: 0.9752 - loss: 0.2758 - val_accuracy: 0.9752 - val_loss: 0.2585
Epoch 5/20
911/911 ━━━━━━━━━━━━━━━━━━━━ 377s 414ms/step - accuracy: 0.9752 - loss: 0.2505 - val_accuracy: 0.9752 - val_loss: 0.2454
Epoch 6/20
911/911 ━━━━━━━━━━━━━━━━━━━━ 378s 415ms/step - accuracy: 0.9752 - loss: 0.2430 - val_accuracy: 0.9752 - val_loss: 0.2418
Epoch 7/20
911/911 ━━━━━━━━━━━━━━━━━━━━ 378s 414ms/step - accuracy: 0.9752 - loss: 0.2412 - val_accuracy: 0.9752 - val_loss: 0.2411
Epoch 8/20
911/911 ━━━━━━━━━━━━━━━━━━━━ 377s 414ms/step - accuracy: 0.9752 -

In [ ]:
test_gen = data_generator(test_images,test_masks, batch_size=16)

accuracy, loss = model.evaluate(test_gen)
print(f'accuracy: {accuracy * 100:.2f}')

   5016/Unknown 1264s 252ms/step - accuracy: 0.9748 - loss: 0.2441